# XTech ETF Flow Factors — v1.0.1

Two ETF flow factors for the eleven S&P 500 sector ETFs, the four books
traded on them, and the daily performance of those books.

The flow is trading in each ETF's **underlying stocks**: institutional and
retail trades in S&P 500 constituents, classified by XTech Flow and aggregated
by GICS sector. ETF subscriptions, redemptions and trading in the ETF shares
themselves are not inputs.

### Key details

- **Coverage**: the 11 GICS sectors of the S&P 500, traded through the SPDR sector ETFs with a SPY hedge
- **Delivery**: daily through the Unifier API, T+1 9:30 AM UTC
- **Data size**: < 1 MB/day across the three tables
- **History**: 2018-09-24 to present

### Datasets

| table | dataset | one row per | `key` |
|---|---|---|---|
| factors | `xtech_us_equity_sector_flow_factors_daily_v1_0_1` | session × release × sector ETF | `etf` |
| weights | `xtech_us_equity_sector_flow_factor_weights_daily_v1_0_1` | session × release × book × ETF | `strategy` |
| pnl | `xtech_us_equity_sector_flow_factor_pnl_daily_v1_0_1` | session × release × book | `strategy` |

The dataset names predate the product name and keep `sector_flow`; they are
the identifiers to query with.

### The four books

| `strategy` | book |
|---|---|
| `inst_impact_reversion63_ls` | beta-neutral long-short on the primary factor |
| `inst_impact_reversion63_beta1` | the same active book as an overlay on one unit of SPY |
| `retail_confirm_drift63_ls` | beta-neutral long-short on the secondary factor |
| `retail_confirm_drift63_beta1` | the same active book as an overlay on one unit of SPY |

### Before querying

- **Every session ships twice**, once per flow classifier release, told apart by the `flow_release` column. This notebook pins
  `v1.2.1` on every query; change `FLOW_RELEASE` below to switch.
- **`asof_date` is the New York trade date.** The data for that date becomes
  available at `asof_datetime`, the next day 04:00 ET. Nothing in a row uses
  information from after its own close.
- Returns are **log** returns, so the `cum_ret_*` columns are sums and
  `drawdown_net` is in the same additive units.

### Install

`nbformat` is what lets Plotly draw inside the notebook; without it every
`fig.show()` below fails.

In [ ]:
#pip install unifier plotly nbformat --upgrade

In [3]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from unifier import unifier

In [ ]:
unifier.user = 'unifier username'
unifier.token = 'unifier api token'
os.environ['UNIFIER_USER'] = unifier.user
os.environ['UNIFIER_TOKEN'] = unifier.token

FACTORS = 'xtech_us_equity_sector_flow_factors_daily_v1_0_1'
WEIGHTS = 'xtech_us_equity_sector_flow_factor_weights_daily_v1_0_1'
PNL     = 'xtech_us_equity_sector_flow_factor_pnl_daily_v1_0_1'

FLOW_RELEASE = 'v1.2.1'                        # or 'v1.2.2'
RELEASE = f"flow_release = '{FLOW_RELEASE}'"   # passed as column_filters on every query

## 1. Query patterns

Every call passes `column_filters=RELEASE`, which pins the flow release in the
warehouse. `column_filters` takes any SQL predicate, so further conditions can
be `AND`ed onto it.

The tables keep one partition per session. `asof_date` returns one session,
and leaving it out returns the latest. For a range use `asof_back_to`, with
`up_to` as the optional upper bound; `back_to` on its own only ever sees the
latest session. Ranges longer than a few months are pulled in pieces, see
section 3.

### Retrieve one session using the asof_date parameter

In [5]:
df = unifier.get_dataframe(name=FACTORS, asof_date='2026-09-02', column_filters=RELEASE)
display(df)

,asof_datetime,flow_release,index_name,gics_sector,etf,inst_impact_reversion63,retail_confirm_drift63,etf_open,etf_high,etf_low,...,n_members_with_flow,flow_coverage_pct,dollar_volume,trade_count,inst_net,inst_gross,retail_net,retail_gross,is_witching,asof_date
0,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Communication Services,XLC,0.058457,-18.595461,111.03,113.0600,110.820,...,24,100,2.988337e+10,2401431,-1.731641e+08,2.820410e+10,3.035800e+07,1.679270e+09,0,2026-09-02
1,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Consumer Discretionary,XLY,0.051013,-20.352988,114.99,115.1100,114.230,...,47,100,3.528210e+10,3319052,-2.908022e+08,3.301869e+10,1.226453e+08,2.263409e+09,0,2026-09-02
2,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Consumer Staples,XLP,0.014531,-13.977487,85.37,86.0800,85.040,...,34,100,1.402373e+10,1857575,-8.971010e+08,1.343236e+10,-1.080470e+07,5.913710e+08,0,2026-09-02
3,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Energy,XLE,-0.017573,-20.423691,64.28,65.3500,64.250,...,21,100,1.105647e+10,1334295,-1.216647e+08,1.059197e+10,2.778267e+07,4.644957e+08,0,2026-09-02
4,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Financials,XLF,-0.022616,-20.237667,57.38,58.2261,57.320,...,76,100,3.493965e+10,3637195,-2.218564e+09,3.356807e+10,-2.515747e+06,1.371584e+09,0,2026-09-02
5,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Health Care,XLV,-0.033761,-16.568191,172.09,174.4900,172.090,...,59,100,2.957940e+10,2677158,-5.621218e+08,2.826349e+10,4.099894e+07,1.315911e+09,0,2026-09-02
6,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Industrials,XLI,-0.008882,-21.684928,173.34,173.3400,171.610,...,83,100,3.138877e+10,3149981,-1.288110e+09,3.004014e+10,-7.301569e+06,1.348623e+09,0,2026-09-02
7,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.070732,-27.068948,183.38,183.7900,182.270,...,73,100,1.835767e+11,11396095,2.923603e+09,1.694227e+11,6.284983e+08,1.415401e+10,0,2026-09-02
8,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Materials,XLB,0.032946,-17.960591,52.36,53.1950,52.270,...,25,100,8.380179e+09,1110959,-1.352813e+08,8.056615e+09,1.069331e+07,3.235642e+08,0,2026-09-02
9,2026-09-03T04:00:00-04:00,v1.2.1,SandP_500,Real Estate,XLRE,-0.003638,-17.884591,44.02,44.0500,43.565,...,30,100,6.264995e+09,920122,-2.706730e+08,6.038809e+09,9.337359e+06,2.261867e+08,0,2026-09-02


### Retrieve one ETF (key='XLK') over a date range using the asof_back_to and up_to parameters

In [6]:
df = unifier.get_dataframe(name=FACTORS, key='XLK', asof_back_to='2026-08-01', up_to='2026-08-31',
                           column_filters=RELEASE)
display(df.sort_values('asof_date'))

,asof_datetime,flow_release,index_name,gics_sector,etf,inst_impact_reversion63,retail_confirm_drift63,etf_open,etf_high,etf_low,...,n_members_with_flow,flow_coverage_pct,dollar_volume,trade_count,inst_net,inst_gross,retail_net,retail_gross,is_witching,asof_date
15,2026-08-04T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.059209,-18.071075,174.010,178.9200,173.0400,...,74,100,2.352698e+11,14555218,-1.240445e+09,2.210840e+11,4.656394e+08,1.418575e+10,0,2026-08-03
16,2026-08-05T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.058953,-19.144532,182.510,187.7050,181.8600,...,74,100,2.867815e+11,16977878,5.078142e+08,2.696766e+11,3.400023e+08,1.710488e+10,0,2026-08-04
1,2026-08-06T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.060742,-19.543877,187.340,188.6200,185.7100,...,74,100,2.315875e+11,14366135,-4.505213e+09,2.171737e+11,4.021475e+08,1.441372e+10,0,2026-08-05
7,2026-08-07T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.060503,-19.394359,183.190,186.9300,182.0600,...,74,100,2.080607e+11,12854811,1.067070e+09,1.941412e+11,5.523619e+08,1.391949e+10,0,2026-08-06
2,2026-08-08T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.059046,-20.776293,188.060,188.7300,185.7300,...,74,100,1.970427e+11,12048227,4.725899e+09,1.842507e+11,2.463762e+08,1.279202e+10,0,2026-08-07
20,2026-08-11T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.061613,-21.999001,188.060,188.8500,186.2600,...,74,100,1.878602e+11,11969615,-3.564387e+09,1.749406e+11,2.787948e+08,1.291956e+10,0,2026-08-10
0,2026-08-12T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.060442,-22.828373,187.800,187.8400,185.0400,...,74,100,1.678095e+11,10636380,4.349582e+09,1.571595e+11,4.005299e+08,1.064997e+10,0,2026-08-11
19,2026-08-13T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.059553,-24.801128,189.470,190.5200,188.3100,...,73,100,2.110940e+11,12910550,2.148563e+09,1.973028e+11,6.435205e+08,1.379123e+10,0,2026-08-12
4,2026-08-14T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.057702,-25.962551,189.130,191.7400,188.8568,...,73,100,2.273646e+11,13204086,3.841948e+09,2.119693e+11,8.634246e+08,1.539532e+10,0,2026-08-13
5,2026-08-15T04:00:00-04:00,v1.2.1,SandP_500,Information Technology,XLK,0.059208,-25.978549,191.140,191.3500,189.2400,...,73,100,1.936413e+11,11064078,-4.531930e+09,1.798433e+11,6.964997e+08,1.379796e+10,0,2026-08-14


### Retrieve one book (key='strategy') for one session from the weights table

In [7]:
df = unifier.get_dataframe(name=WEIGHTS, key='inst_impact_reversion63_beta1', asof_date='2026-09-02',
                           column_filters=RELEASE)
display(df)

,asof_datetime,flow_release,strategy,factor,gics_sector,etf,held_weight,target_weight,beta,beta_contribution,signal_z,factor_value,resid_vol_63,asof_date
0,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,(market hedge),SPY,0.630899,NaN,1.000000,0.630899,NaN,NaN,NaN,2026-09-02
1,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,Communication Services,XLC,0.093456,0.184627,0.684588,0.063979,1.248778,0.059371,0.011708,2026-09-02
2,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,Consumer Discretionary,XLY,0.232062,0.220955,1.151055,0.267116,1.009915,0.050835,0.009783,2026-09-02
3,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,Consumer Staples,XLP,0.040688,0.000000,-0.059645,-0.002427,-0.051280,0.012914,0.010881,2026-09-02
4,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,Energy,XLE,-0.025008,-0.158467,-0.303426,0.007588,-0.902101,-0.017490,0.013649,2026-09-02
5,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,Financials,XLF,-0.167886,-0.240564,0.634950,-0.106599,-1.090365,-0.024218,0.008991,2026-09-02
6,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,Health Care,XLV,0.026726,-0.166263,0.278526,0.007444,-1.357406,-0.033760,0.013009,2026-09-02
7,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,Industrials,XLI,-0.128006,-0.235449,0.949736,-0.121572,-0.680174,-0.009560,0.009187,2026-09-02
8,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,Information Technology,XLK,0.109773,0.175850,1.718106,0.188601,1.613640,0.072409,0.012292,2026-09-02
9,2026-09-03T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,Materials,XLB,0.186576,0.195004,0.775271,0.144647,0.492283,0.032338,0.011085,2026-09-02


### Retrieve one book's daily P&L over a date range

In [8]:
df = unifier.get_dataframe(name=PNL, key='inst_impact_reversion63_beta1', asof_back_to='2026-08-01',
                           up_to='2026-08-31', column_filters=RELEASE)
display(df.sort_values('asof_date'))

,asof_datetime,flow_release,strategy,factor,factor_sign,beta_target,lam,ret_gross,ret_net,ret_active,...,spy_leg,rolling_vol_63,rolling_vol_252,rolling_sharpe_63,rolling_sharpe_252,rolling_sortino_252,rolling_beta_252,drawdown_net,max_drawdown_net,asof_date
12,2026-08-04T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,0.012642,0.012611,-0.000863,...,0.846363,0.139524,0.147256,2.612718,1.979188,2.952106,0.982602,-0.000286,-0.291997,2026-08-03
11,2026-08-05T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,0.016750,0.016732,-0.001927,...,0.836056,0.141374,0.146908,3.287950,2.215373,3.348269,0.979005,0.000000,-0.291997,2026-08-04
6,2026-08-06T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,0.005600,0.005579,0.008013,...,0.825506,0.141031,0.146501,3.227782,2.170841,3.271867,0.978916,0.000000,-0.291997,2026-08-05
15,2026-08-07T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,-0.001734,-0.001754,-0.000343,...,0.814065,0.139339,0.146319,2.842620,2.206819,3.329146,0.978229,-0.001754,-0.291997,2026-08-06
20,2026-08-08T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,0.008923,0.008903,0.003371,...,0.801506,0.138113,0.146034,3.412727,2.182008,3.285307,0.976892,0.000000,-0.291997,2026-08-07
17,2026-08-11T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,0.002560,0.002528,0.002586,...,0.790509,0.137938,0.145804,3.536681,2.250686,3.392185,0.976587,0.000000,-0.291997,2026-08-10
8,2026-08-12T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,-0.004845,-0.004866,-0.001450,...,0.778853,0.138473,0.145643,3.242948,2.147965,3.229711,0.975959,-0.004866,-0.291997,2026-08-11
14,2026-08-13T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,-0.003232,-0.003252,-0.006368,...,0.767175,0.138627,0.145712,2.987554,2.113037,3.176898,0.975726,-0.008118,-0.291997,2026-08-12
13,2026-08-14T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,0.001877,0.001857,-0.004507,...,0.755631,0.138619,0.145293,2.971157,2.047687,3.069801,0.973176,-0.006261,-0.291997,2026-08-13
9,2026-08-15T04:00:00-04:00,v1.2.1,inst_impact_reversion63_beta1,INST_IMPACT_REVERSION63,1,1,1,-0.004425,-0.004445,-0.002237,...,0.745143,0.138834,0.145380,2.661079,1.990497,2.982708,0.973747,-0.010706,-0.291997,2026-08-14


### Download all dataset in daily P&L via replicate features

In [ ]:
unifier.replicate(
    PNL,
    target_location="./"
)

Found 1 files to download.
Native replication completed for xtech_us_equity_sector_flow_factor_pnl_daily_v1_0_1. Downloaded 1 files.


In [ ]:
unifier.replicate(
    "xtech_us_equity_sector_flow_factor_weights_daily_v1_0_1",
    target_location="./",
    asof_date="2019-02-21",
)


In [ ]:


unifier.replicate(
    "xtech_us_equity_sector_flow_factors_daily_v1_0_1",
    target_location="./"
)

## 2. What is in the tables

`get_dataset_details` returns the catalog entry, including the column
dictionary a consumer sees on the site.

In [9]:
details = unifier.get_dataset_details(name=FACTORS)
dictionary = pd.DataFrame(details['data_dictionary']['data_dictionary'])
display(dictionary[['column_name', 'data_type', 'description']])

,column_name,data_type,description
0,asof_date,date,"New York trade date the row describes, formatt..."
1,asof_datetime,varchar,ISO-8601 New York timestamp at which this sess...
2,flow_release,varchar,Version of the upstream ETI equity-flow releas...
3,index_name,varchar,Index whose sector membership defines the pane...
4,gics_sector,varchar,"GICS sector name, one of the eleven sectors of..."
5,etf,varchar,"The SPDR ETF standing in for the sector (XLB, ..."
6,INST_IMPACT_REVERSION63,double,"Institutional impact reversion, accumulated ov..."
7,RETAIL_CONFIRM_DRIFT63,double,"Retail confirmation drift, accumulated over 63..."
8,etf_open,double,"Sector ETF opening price for the session, spli..."
9,etf_high,double,"Sector ETF session high, adjusted."


## 3. Pull the working set

All four books' P&L since the first session a book holds, and two years of
weights and factors for the heatmaps. Everything below is drawn from these
frames.

A single query over years of sessions currently fails on the service, so
`pull` asks for one 3-month window at a time and stitches them together. A
failed request comes back from `get_dataframe` as an empty frame rather than an
error, and every window holds sessions, so an empty window is retried. The
full pull takes several minutes.

In [10]:
def pull(name, since, key=None, attempts=10):
    parts = []
    since, hi = pd.Timestamp(since), pd.Timestamp.today().normalize()
    while hi >= since:
        lo = max(hi - pd.DateOffset(months=3) + pd.Timedelta(days=1), since)
        for _ in range(attempts):
            df = unifier.get_dataframe(name=name, key=key, asof_back_to=f'{lo:%Y-%m-%d}',
                                       up_to=f'{hi:%Y-%m-%d}', column_filters=RELEASE)
            if not df.empty:
                break
        else:
            raise RuntimeError(f'{name} {lo:%Y-%m-%d}..{hi:%Y-%m-%d}: no rows after {attempts} attempts')
        parts.append(df)
        hi = lo - pd.Timedelta(days=1)
    df = pd.concat(parts, ignore_index=True)
    df['asof_date'] = pd.to_datetime(df['asof_date'])
    return df.sort_values('asof_date', ignore_index=True)


STRATEGY = 'inst_impact_reversion63_beta1'
FACTOR   = 'inst_impact_reversion63'

allbooks = pull(PNL, since='2019-02-22')                  # all four books
wts      = pull(WEIGHTS, since='2024-09-01', key=STRATEGY)
fac      = pull(FACTORS, since='2024-09-01')
pnl      = allbooks[allbooks.strategy == STRATEGY]

print(f'pnl     {len(allbooks):>6,} rows  {allbooks.asof_date.min():%Y-%m-%d} .. {allbooks.asof_date.max():%Y-%m-%d}  (four books)')
print(f'weights {len(wts):>6,} rows  {wts.asof_date.min():%Y-%m-%d} .. {wts.asof_date.max():%Y-%m-%d}')
print(f'factors {len(fac):>6,} rows  {fac.asof_date.min():%Y-%m-%d} .. {fac.asof_date.max():%Y-%m-%d}')

pnl      7,644 rows  2019-02-22 .. 2026-09-29  (four books)
weights  6,240 rows  2024-09-03 .. 2026-09-29
factors  5,718 rows  2024-09-03 .. 2026-09-29


### Plot styling

Lifted from the product's own monitor, so the charts below are the ones a
consumer would see there.

In [11]:
PLOTLY_LAYOUT = dict(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(family='IBM Plex Sans, ui-sans-serif, system-ui, sans-serif',
              size=13, color='#4a5361'),
    margin=dict(l=56, r=16, t=36, b=40),
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0),
)
BLUE, RED, GREY = '#2a78d6', '#e34948', '#94a3b8'
ETF_COLOR = {
    'XLC': '#6366f1', 'XLY': '#f59e0b', 'XLP': '#10b981', 'XLE': '#ef4444',
    'XLF': '#8b5cf6', 'XLV': '#06b6d4', 'XLI': '#ec4899', 'XLK': '#3b82f6',
    'XLB': '#a16207', 'XLRE': '#14b8a6', 'XLU': '#64748b', 'SPY': '#111827',
}
ETF_ORDER = ['XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLK',
             'XLB', 'XLRE', 'XLU']
STRATEGY_LABEL = {
    'inst_impact_reversion63_ls':     'Impact reversion · LS (beta=0)',
    'inst_impact_reversion63_beta1':  'Impact reversion · overlay (beta=1)',
    'retail_confirm_drift63_ls':      'Retail confirm · LS (beta=0)',
    'retail_confirm_drift63_beta1':   'Retail confirm · overlay (beta=1)',
}
FACTOR_LABEL = {
    'inst_impact_reversion63': 'Inst. impact reversion (63d)',
    'retail_confirm_drift63':  'Retail confirm drift (63d)',
}


def styled(fig, **extra):
    fig.update_layout(**{**PLOTLY_LAYOUT, **extra})
    fig.update_xaxes(showgrid=True, gridcolor='rgba(74,83,97,0.12)', zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(74,83,97,0.12)', zeroline=False)
    return fig

## 4. Wealth

Net, gross and the SPY benchmark for one book. `1 + cum_ret_net` reads as
wealth because the cumulative columns are sums of log returns.

In [12]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=pnl.asof_date, y=1.0 + pnl.cum_ret_net, name='net',
                         mode='lines', line=dict(color=BLUE, width=1.8)))
fig.add_trace(go.Scatter(x=pnl.asof_date, y=1.0 + pnl.cum_ret_gross, name='gross',
                         mode='lines', line=dict(color=BLUE, width=1.1, dash='dot')))
fig.add_trace(go.Scatter(x=pnl.asof_date, y=1.0 + pnl.cum_ret_bench, name='SPY',
                         mode='lines', line=dict(color=GREY, width=1.1)))
fig.update_yaxes(title='wealth')
styled(fig, title=f'{STRATEGY_LABEL[STRATEGY]} — {FLOW_RELEASE}', height=460).show()

## 5. Trailing risk

Sharpe and Sortino share a panel because they answer the same question with a
different denominator; the drawdown panel is filled so the depth reads at a
glance rather than from the axis.

In [13]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=('rolling Sharpe 252', 'rolling vol 252', 'drawdown (net)'))
fig.add_trace(go.Scatter(x=pnl.asof_date, y=pnl.rolling_sharpe_252, name='Sharpe',
                         mode='lines', line=dict(color=BLUE, width=1.6)), row=1, col=1)
fig.add_trace(go.Scatter(x=pnl.asof_date, y=pnl.rolling_sortino_252, name='Sortino',
                         mode='lines', line=dict(color=BLUE, width=1.2, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=pnl.asof_date, y=pnl.rolling_vol_252, name='vol', showlegend=False,
                         mode='lines', line=dict(color=BLUE, width=1.6)), row=2, col=1)
fig.add_trace(go.Scatter(x=pnl.asof_date, y=pnl.drawdown_net, name='drawdown', showlegend=False,
                         mode='lines', line=dict(color=RED, width=1.4), fill='tozeroy',
                         fillcolor='rgba(239,68,68,0.12)'), row=3, col=1)
fig.update_yaxes(title_text='Sharpe / Sortino', row=1, col=1)
fig.update_yaxes(title_text='vol', tickformat='.1%', row=2, col=1)
fig.update_yaxes(title_text='dd', tickformat='.1%', row=3, col=1)
styled(fig, height=720, legend=dict(orientation='h', yanchor='bottom', y=1.04, x=0)).show()

## 6. What the book costs to run

Turnover is what the cost line is charged on, gross exposure is what the book
holds, and the SPY leg is the market unit the overlay carries. For a `_beta1`
book the SPY leg sits near +1; for the `_ls` variant it is the hedge and sits
near zero.

In [14]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=('turnover (ann.)', 'gross exposure', 'SPY hedge leg'))
fig.add_trace(go.Scatter(x=pnl.asof_date, y=pnl.turnover_ann, mode='lines',
                         line=dict(color=BLUE, width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=pnl.asof_date, y=pnl.gross_exposure, mode='lines',
                         line=dict(color=BLUE, width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=pnl.asof_date, y=pnl.spy_leg, mode='lines',
                         line=dict(color=BLUE, width=1.5)), row=3, col=1)
fig.add_hline(y=0, line_color='rgba(74,83,97,0.35)', row=3, col=1)
fig.update_yaxes(title_text='x / year', row=1, col=1)
fig.update_yaxes(title_text='gross', row=2, col=1)
fig.update_yaxes(title_text='SPY w', row=3, col=1)
styled(fig, height=680, showlegend=False).show()

## 7. The positions themselves

The weights table is the book, sector by sector, day by day. Red is short,
blue is long; the SPY leg is drawn underneath because it is on a different
scale from the sector bets and would flatten them if it shared the colour axis.

In [15]:
sector = wts[wts.etf != 'SPY']
spy = wts[wts.etf == 'SPY']

pivot = (sector.pivot_table(index='etf', columns='asof_date', values='held_weight', aggfunc='mean')
               .reindex(ETF_ORDER))
lim = max(float(np.nanpercentile(np.abs(pivot.to_numpy()), 99.5)), 1e-6)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    row_heights=[0.78, 0.22],
                    subplot_titles=('held sector weight', 'SPY hedge leg'))
fig.add_trace(go.Heatmap(z=pivot.to_numpy(), x=pivot.columns, y=list(pivot.index),
                         colorscale='RdBu', zmid=0, zmin=-lim, zmax=lim,
                         colorbar=dict(len=0.72, y=0.62, thickness=12),
                         hovertemplate='%{y} %{x|%Y-%m-%d}<br>w=%{z:.3f}<extra></extra>'),
              row=1, col=1)
fig.add_trace(go.Scatter(x=spy.asof_date, y=spy.held_weight, mode='lines', name='SPY',
                         line=dict(color=ETF_COLOR['SPY'], width=1.4),
                         hovertemplate='SPY %{x|%Y-%m-%d}<br>w=%{y:.3f}<extra></extra>'),
              row=2, col=1)
fig.update_yaxes(autorange='reversed', row=1, col=1)
fig.update_yaxes(title_text='SPY w', row=2, col=1)
styled(fig, height=640, hovermode='closest').show()

### The book as of the latest session

A sanity check worth running on any day you pull: for a `_beta1` book the
sector weights net to roughly zero and `beta_contribution` sums to the beta
target.

In [16]:
day = wts.asof_date.max()
snap = sector[sector.asof_date == day].copy()
snap['etf'] = pd.Categorical(snap['etf'], ETF_ORDER, ordered=True)
snap = snap.sort_values('etf')

fig = go.Figure(go.Bar(
    x=snap.etf.astype(str), y=snap.held_weight,
    marker_color=[BLUE if v >= 0 else RED for v in snap.held_weight],
    hovertemplate='%{x}<br>held=%{y:.3f}<extra></extra>'))
fig.add_hline(y=0, line_color='rgba(74,83,97,0.4)')
fig.update_yaxes(title='held weight')
styled(fig, title=f'book on {day:%Y-%m-%d}  ({FLOW_RELEASE})', hovermode='x', height=380).show()

print(f'sector weights sum to   {snap.held_weight.sum():+.4f}')
print(f'beta contributions sum  {wts[wts.asof_date == day].beta_contribution.sum():.4f}  (target 1)')

sector weights sum to   +0.0000
beta contributions sum  1.0000  (target 1)


## 8. The factor underneath

`inst_impact_reversion63` is the negative of a sector's accumulated
institutional price impact over the trailing quarter, so a **high** score is a
sector expected to give back less — the sign is folded into the definition and
higher always means hold it long.

The cross-section is what the book trades; the heatmap is how persistent those
bets are.

In [17]:
day = fac.asof_date.max()
snap = fac[fac.asof_date == day].sort_values(FACTOR, ascending=False)

fig = go.Figure(go.Bar(
    x=snap.etf, y=snap[FACTOR],
    marker_color=[BLUE if v >= 0 else RED for v in snap[FACTOR]],
    customdata=snap.gics_sector,
    hovertemplate='%{x} %{customdata}<br>%{y:.3f}<extra></extra>'))
fig.add_hline(y=0, line_color='rgba(74,83,97,0.4)')
fig.update_yaxes(title=FACTOR_LABEL[FACTOR])
styled(fig, title=f'{FACTOR_LABEL[FACTOR]} on {day:%Y-%m-%d}', hovermode='x', height=380).show()

In [18]:
pivot = (fac.pivot_table(index='etf', columns='asof_date', values=FACTOR, aggfunc='mean')
            .reindex(ETF_ORDER))
lim = max(float(np.nanpercentile(np.abs(pivot.to_numpy()), 98)), 1e-6)

fig = go.Figure(go.Heatmap(z=pivot.to_numpy(), x=pivot.columns, y=list(pivot.index),
                           colorscale='RdBu', zmid=0, zmin=-lim, zmax=lim,
                           colorbar=dict(thickness=12),
                           hovertemplate='%{y} %{x|%Y-%m-%d}<br>%{z:.3f}<extra></extra>'))
styled(fig, height=420, hovermode='closest', title=f'{FACTOR_LABEL[FACTOR]} — {FLOW_RELEASE}').show()

## 9. Coverage

`flow_coverage_pct` is the share of a sector's index members that carried
usable flow that session. It is the honest health metric for this product: the
factor is a member-weighted aggregate, so a session where coverage falls is a
session where the score is built on fewer names, whatever its value looks like.

In [19]:
cov = fac.groupby('asof_date')['flow_coverage_pct'].mean()

fig = go.Figure(go.Scatter(x=cov.index, y=cov, mode='lines', line=dict(color=BLUE, width=1.4)))
fig.update_yaxes(title='mean flow coverage %', range=[0, 105])
styled(fig, height=320, title='member coverage').show()

## 10. The sector tape itself

The factors table carries the ETF bar and the benchmark close alongside the
scores, so a rebased sector chart needs no second dataset.

In [20]:
px = (fac.pivot_table(index='asof_date', columns='etf', values='etf_close', aggfunc='mean')
         .reindex(columns=ETF_ORDER)
         .dropna(how='all'))
bench = fac.drop_duplicates('asof_date').set_index('asof_date')['bench_close'].dropna()

fig = go.Figure()
for etf in px.columns:
    fig.add_trace(go.Scatter(x=px.index, y=px[etf] / px[etf].iloc[0] * 100.0, name=etf,
                             mode='lines', line=dict(color=ETF_COLOR[etf], width=1.4)))
fig.add_trace(go.Scatter(x=bench.index, y=bench / bench.iloc[0] * 100.0, name='S&P 500',
                         mode='lines', line=dict(color=GREY, width=1.6, dash='dot')))
fig.update_yaxes(title='adjusted close (rebased to 100)')
styled(fig, height=460, title='sector ETFs, rebased').show()

## 11. Comparing the four books

The pnl table holds all four books, and the working set pulled it with no
`key`, so a like-for-like table needs no further query.

In [21]:
rows = []
for name, g in allbooks.groupby('strategy'):
    mu, sd = g.ret_net.mean(), g.ret_net.std()
    rows.append({
        'strategy': STRATEGY_LABEL.get(name, name),
        'sessions': f'{len(g):,}',
        'ann. return': f'{mu * 252:+.2%}',
        'ann. vol': f'{sd * np.sqrt(252):.2%}',
        'Sharpe': f'{mu / sd * np.sqrt(252):.2f}',
        'max drawdown': f'{g.drawdown_net.min():.2%}',
        'turnover (ann.)': f'{g.turnover_ann.iloc[-1]:.1f}x',
        'cum net': f'{g.cum_ret_net.iloc[-1]:+.2%}',
    })
display(pd.DataFrame(rows).set_index('strategy'))

,sessions,ann. return,ann. vol,Sharpe,max drawdown,turnover (ann.),cum net
strategy,,,,,,,
Impact reversion · overlay (beta=1),"1,911",+23.55%,19.13%,1.23,-29.20%,4.7x,+178.22%
Impact reversion · LS (beta=0),"1,911",+8.89%,6.85%,1.30,-9.43%,4.7x,+67.33%
Retail confirm · overlay (beta=1),"1,911",+15.79%,19.76%,0.80,-38.14%,4.7x,+119.41%
Retail confirm · LS (beta=0),"1,911",+1.21%,7.00%,0.17,-15.37%,4.7x,+9.20%


In [22]:
fig = go.Figure()
palette = ['#2a78d6', '#eb6834', '#10b981', '#8b5cf6']
for color, (name, g) in zip(palette, allbooks.groupby('strategy')):
    fig.add_trace(go.Scatter(x=g.asof_date, y=1.0 + g.cum_ret_net,
                             name=STRATEGY_LABEL.get(name, name), mode='lines',
                             line=dict(color=color, width=1.6)))
fig.update_yaxes(title='wealth (net)')
styled(fig, height=460, title=f'all four books — {FLOW_RELEASE}').show()

---

**Terms.** The four books are published as a description of what the factors
do, not as investment advice. Costs are a round-trip commission and slippage
assumption in basis points of notional traded; they exclude financing and
borrow on the short leg, which a live short-sector book would pay.